# TGLC 200s Sanity Check

Exploratory notebook verifying that TGLC (Sector 56+, 200s cadence) is a viable replacement for SPOC 2-min as the primary light curve source.
Checks availability, cadence, light curve quality, and window re-parameterization.

---
## Section 0: Imports & Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import time
import threading
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astroquery.mast import Catalogs, Observations
import lightkurve as lk

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# astroquery socket-level timeout
Observations.TIMEOUT = 20

# --- Project root detection (walk up until CLAUDE.md is found) ---
project_root = r'c:\git_repo\Stellar-World-Model'

print(f'Project root: {project_root}')

Project root: c:\git_repo\Stellar-World-Model


---
## Section 1: Load Sample

Load `processed/df_final.csv` (6,654 bright nearby stars, Tmag < 7, plx > 10 mas).
Randomly sample 20 stars for the TGLC availability check.

In [3]:
df_final = pd.read_csv(project_root / 'processed' / 'df_final.csv')
print(f'df_final shape: {df_final.shape}')
print()
print(df_final.head(5).to_string(index=False))

sample20 = df_final.sample(n=20, random_state=42).reset_index(drop=True)
tic_ids_20 = sample20['ID'].astype(int).tolist()

print(f'\n20 randomly sampled TIC IDs (random_state=42):')
print(tic_ids_20)

df_final shape: (6654, 13)

       ID   Tmag    Teff    logg        MH     plx      rad  mass  GAIAmag  gaiaBp  gaiaRp    TIC_ID  n_sectors
117927293 5.9418 6384.00 3.81453       NaN 16.5095 2.319380  1.28  6.30145 6.58443 5.90321 117927293          2
117956546 5.0430 5365.25     NaN -0.844312 79.0696 0.866258   NaN  5.52379 5.93117 5.00820 117956546          2
373061126 6.3452 7133.52 4.03529       NaN 14.2137 2.004870  1.59  6.46584 6.57503 6.32484 373061126          5
238432056 5.3971 5489.00 4.52192  0.070000 72.5764 0.889627  0.96  5.87245 6.26563 5.35394 238432056          3
395318337 6.7920 7173.00 4.16377       NaN 12.6336 1.734650  1.60  7.01269 7.18784 6.76387 395318337          1

20 randomly sampled TIC IDs (random_state=42):
[39954008, 422679659, 65628544, 189113383, 136947179, 279265273, 49845357, 63584475, 437327038, 415843217, 411218105, 353051540, 73434673, 284292958, 10632269, 284472887, 51750453, 800131628, 188480881, 351534793]


---
## Section 2: TGLC 200s Availability Check

For each of the 20 sampled TIC IDs, query MAST for TGLC light curves and filter to Sector 56+ (200s cadence, Cycles 5+).
Uses the same `search_with_timeout` + MAST session reset pattern as the reference notebook.

In [4]:
# --- Constants ---
QUERY_TIMEOUT  = 20    # astroquery socket-level cap (seconds)
THREAD_TIMEOUT = 25    # daemon thread cap (fires only if astroquery's own timeout misses)
QUERY_DELAY    = 0.5   # politeness sleep between queries
MIN_SECTOR     = 56    # first sector with 200s FFI cadence


def _reset_mast_session():
    """Drop astroquery's HTTP session so dangling connections are closed."""
    try:
        sess = getattr(Observations, '_session', None)
        if sess is not None:
            sess.close()
    except Exception:
        pass
    try:
        Observations._session = requests.Session()
    except Exception:
        pass


def search_tglc_with_timeout(tic_id: int, timeout: int = THREAD_TIMEOUT):
    """Return (n_sectors_56plus, sector_list_56plus, status).
    status in {ok, timeout, error:<typename>}.
    """
    result = [0, []]
    exc = [None]

    def _run():
        try:
            sr = lk.search_lightcurve(f'TIC {tic_id}', mission='TESS', author='TGLC')
            if len(sr) == 0:
                result[0], result[1] = 0, []
                return
            try:
                all_sectors = list(sr.table['sequence_number'])
            except Exception:
                all_sectors = []
            sectors_56plus = [s for s in all_sectors if int(s) >= MIN_SECTOR]
            result[0] = len(sectors_56plus)
            result[1] = sectors_56plus
        except Exception as e:
            exc[0] = e

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout)

    if t.is_alive():
        return 0, [], 'timeout'
    if exc[0] is not None:
        name = type(exc[0]).__name__
        if 'Timeout' in name or 'ConnectionError' in name:
            return 0, [], 'timeout'
        return 0, [], f'error:{name}'
    return result[0], result[1], 'ok'

In [5]:
# --- MAST warmup sanity check ---
WARMUP_TIC = tic_ids_20[0]
print(f'MAST warmup on TIC {WARMUP_TIC}...')
t_warm = time.time()
n_warm, sec_warm, status_warm = search_tglc_with_timeout(WARMUP_TIC)
dt_warm = time.time() - t_warm
print(f'  status={status_warm}  n_sectors_56plus={n_warm}  took {dt_warm:.1f}s')
if status_warm not in ('ok',):
    raise RuntimeError(f'MAST warmup failed: {status_warm}. Check https://outerspace.stsci.edu/display/MASTDOCS')
if dt_warm > 5:
    print(f'  Warning: MAST is responding but slowly ({dt_warm:.1f}s). Loop will be slow.')

# --- Main loop over 20-star sample ---
print(f'\nQuerying TGLC availability for 20 stars (Sector >= {MIN_SECTOR})...')
print(f'{"-"*70}')

avail_records = []
consec_bad = 0
t0 = time.time()

for i, tic_id in enumerate(tic_ids_20, start=1):
    t_q = time.time()
    n, sectors, status = search_tglc_with_timeout(tic_id)
    dt = time.time() - t_q

    if status == 'ok':
        consec_bad = 0
        avail_records.append({
            'tic_id': tic_id,
            'n_tglc_sectors_56plus': n,
            'sector_list_56plus': sectors,
        })
        print(f'  [{i:>2}/20]  TIC {tic_id:>11}  n_56plus={n:>3}  {dt:>5.1f}s  sectors={sectors}')
    else:
        consec_bad += 1
        # Do NOT append — unknown answer; re-run cell to retry this star
        if status == 'timeout':
            _reset_mast_session()
            backoff = min(5 * (2 ** min(consec_bad - 1, 4)), 60)
            print(f'  [{i:>2}/20]  TIC {tic_id:>11}  TIMEOUT — reset session, back off {backoff}s')
            time.sleep(backoff)
        else:
            print(f'  [{i:>2}/20]  TIC {tic_id:>11}  {status}')

    if i < 20:
        time.sleep(QUERY_DELAY)

print(f'{"-"*70}')
print(f'Done in {time.time() - t0:.1f}s  ({len(avail_records)}/20 queries recorded)')

df_avail = pd.DataFrame(avail_records)

MAST warmup on TIC 39954008...


No data found for target "TIC 39954008".


  status=ok  n_sectors_56plus=0  took 4.7s

Querying TGLC availability for 20 stars (Sector >= 56)...
----------------------------------------------------------------------
  [ 1/20]  TIC    39954008  n_56plus=  0    0.0s  sectors=[]
  [ 2/20]  TIC   422679659  n_56plus=  0    0.6s  sectors=[]


No data found for target "TIC 65628544".


  [ 3/20]  TIC    65628544  n_56plus=  0    3.5s  sectors=[]


...

  [18/20]  TIC   800131628  n_56plus=  0    3.3s  sectors=[]


No data found for target "TIC 188480881".


  [19/20]  TIC   188480881  n_56plus=  0    2.7s  sectors=[]


No data found for target "TIC 351534793".


  [20/20]  TIC   351534793  n_56plus=  0    2.5s  sectors=[]
----------------------------------------------------------------------
Done in 57.5s  (20/20 queries recorded)
